# Lab 3.2: A Leakage Hunt in Hospital Readmission Data

Run the cells from top to bottom. You do not need to change any code until the optional last section. Each checkpoint tells you what to look for.

Data: Diabetes 130-US Hospitals for Years 1999-2008 (Strack et al., 2014), UCI Machine Learning Repository, CC BY 4.0. 101,766 hospital encounters of patients with diabetes. Target: readmitted within 30 days of discharge.

## 1. Load the data

In [1]:
import zipfile, urllib.request, tempfile
from pathlib import Path
import numpy as np, pandas as pd

UCI = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"

def load_readmission():
    """Diabetes 130-US Hospitals (UCI, CC BY 4.0). Downloads once to a temporary folder;
    if you are offline, put diabetic_data.csv next to this notebook."""
    local = Path("diabetic_data.csv")
    if not local.exists():
        cache = Path(tempfile.gettempdir()) / "xai-ch03"
        cache.mkdir(exist_ok=True)
        local = cache / "diabetic_data.csv"
        if not local.exists():
            z = cache / "d.zip"
            urllib.request.urlretrieve(UCI, z)
            zipfile.ZipFile(z).extractall(cache)
    d = pd.read_csv(local)
    d["y"] = (d["readmitted"] == "<30").astype(int)          # target: back within 30 days
    d["age_n"] = d["age"].str.extract(r"\[(\d+)").astype(int)  # "[70-80)" -> 70
    for c in ["A1Cresult", "insulin", "change", "diabetesMed", "gender", "max_glu_serum"]:
        d[c + "_c"] = d[c].astype("category").cat.codes
    return d.sort_values("encounter_id").reset_index(drop=True)

NUM = ["age_n", "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["admission_type_id", "discharge_disposition_id", "admission_source_id", "A1Cresult_c",
       "insulin_c", "change_c", "diabetesMed_c", "gender_c", "max_glu_serum_c"]
d = load_readmission()
print(d.shape)

(101766, 58)


## 2. Look before you model

**Checkpoint 1.** How many encounters, how many distinct patients, how many patients appear more than once, and what share of encounters end in a 30-day readmission?

In [2]:
print('encounters:', len(d))
print('patients:  ', d['patient_nbr'].nunique())
print('patients with more than one encounter:', (d.groupby('patient_nbr').size() > 1).sum())
print('30-day readmission rate: %.3f' % d['y'].mean())

encounters: 101766
patients:   71518
patients with more than one encounter: 16773
30-day readmission rate: 0.112


**Checkpoint 2.** Discharge disposition codes 11, 19 and 20 mean the patient died. What is their readmission rate? Why is that not a finding about medicine?

In [3]:
died = d['discharge_disposition_id'].isin([11, 19, 20])
print('encounters ending in death:', died.sum(), ' readmission rate among them:', d.loc[died, 'y'].mean())

encounters ending in death: 1652  readmission rate among them: 0.0


## 3. An honest model, two ways of splitting

The features below are all known at discharge. We train a gradient-boosted tree model twice: once with a random split of encounters, once with a split that keeps every patient on one side.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def split(data, how, seed=0):
    idx = np.arange(len(data))
    if how == "rows":
        return train_test_split(idx, test_size=0.3, random_state=seed, stratify=data["y"])
    return next(GroupShuffleSplit(1, test_size=0.3, random_state=seed).split(idx, groups=data["patient_nbr"]))

def fit(data, feats, how):
    tr, te = split(data, how)
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0,
                                       categorical_features=[f in CAT for f in feats])
    m.fit(data[feats].iloc[tr], data["y"].iloc[tr])
    p = m.predict_proba(data[feats].iloc[te])[:, 1]
    overlap = len(set(data["patient_nbr"].iloc[tr]) & set(data["patient_nbr"].iloc[te]))
    return m, tr, te, p, roc_auc_score(data["y"].iloc[te], p), overlap

BASE = NUM + CAT
for how in ['rows', 'patients']:
    *_, auc, ov = fit(d, BASE, how)
    print(f'split by {how:8s}: test AUC {auc:.3f}; patients in both train and test: {ov}')

split by rows    : test AUC 0.673; patients in both train and test: 8993


split by patients: test AUC 0.677; patients in both train and test: 0


**Checkpoint 3.** The random split puts several thousand patients on both sides. Here the two AUCs are close, because none of these features lets the model recognize an individual patient. Do not conclude that grouping never matters: with images, free text or detailed histories, a model can recognize the person, and the random split then rewards memory.

## 4. Plant the leak that feature engineering often creates

A common enrichment step counts how many encounters each patient has. Computed over the whole file, the count includes encounters that happen *after* the one being predicted. A past-only version counts earlier encounters only.

In [5]:
d['encounters_in_file'] = d.groupby('patient_nbr')['encounter_id'].transform('count')  # leaky
d['prior_encounters'] = d.groupby('patient_nbr').cumcount()                              # past only
for extra in ['prior_encounters', 'encounters_in_file']:
    m, tr, te, p, auc, ov = fit(d, BASE + [extra], 'patients')
    print(f'+ {extra:20s}: test AUC {auc:.3f}')

+ prior_encounters    : test AUC 0.680


+ encounters_in_file  : test AUC 0.793


**Checkpoint 4.** The leaky feature lifts test AUC from about 0.68 to about 0.79, even with the patient-grouped split. The test set cannot catch this leak, because the leak is inside every row.

## 5. Let an explanation find the leak

Permutation importance shuffles one feature at a time in the test set and measures how much the AUC drops. Chapter 6 covers the method in depth; here we use it as a smoke detector.

In [6]:
from sklearn.inspection import permutation_importance
feats = BASE + ['encounters_in_file']
m, tr, te, p, auc, ov = fit(d, feats, 'patients')
r = permutation_importance(m, d[feats].iloc[te], d['y'].iloc[te], scoring='roc_auc', n_repeats=3, random_state=0)
imp = pd.Series(r.importances_mean, index=feats).sort_values(ascending=False)
print(imp.head(6).round(3))

encounters_in_file          0.207
number_inpatient            0.053
discharge_disposition_id    0.039
admission_type_id           0.005
admission_source_id         0.004
num_lab_procedures          0.002
dtype: float64


**Checkpoint 5.** `encounters_in_file` should top the list with a drop of about 0.2, roughly four times the next feature. A single engineered feature dominating a weak clinical signal is the classic signature of leakage. Ask of every top feature: could I have known this value at the moment of prediction?

## 6. Define the population the way the original study did

Strack et al. (2014) kept only the first encounter of each patient and removed encounters that ended in death or discharge to hospice (codes 11, 13, 14, 19, 20, 21).

In [7]:
pop = d[~d['discharge_disposition_id'].isin([11, 13, 14, 19, 20, 21])]
pop = pop.groupby('patient_nbr').head(1).reset_index(drop=True)
print('encounters kept:', len(pop), ' readmission rate: %.3f' % pop['y'].mean())
*_, auc, ov = fit(pop, BASE, 'rows')
print(f'honest model on this population: test AUC {auc:.3f}')

encounters kept: 69990  readmission rate: 0.090


honest model on this population: test AUC 0.647


**Checkpoint 6.** About 70,000 encounters remain (the paper reports 69,984; small differences come from exclusion details), the readmission rate falls to about 9 percent and the AUC to about 0.65. Which number would you report, and for which use?

## 7. Optional: your own leak

Invent one more feature that a well-meaning analyst might build (for example, the number of distinct medications a patient ever received in the file) and test whether permutation importance flags it.

In [8]:
# your code here (optional)